In [ ]:
import pymupdf
import json
import csv
import pandas as pd
import glob
import os
import time
import re
from functools import partial
from dotenv import load_dotenv
import concurrent.futures

# ==================== GELIŞTIRILMIŞ QA_ROLE ====================
QA_role = """
# GÖREV TANIMI (PRIMARY ROLE DEFINITION):
Sen, Türk Tüketici Hukuku alanında uzmanlaşmış bir yapay zeka hukuk asistanısın.
Amacın, kamuoyunu bilgilendirmek için, verilen yasal metinlerden (Kanun, Yönetmelik, Makale) Türkçe Soru-Cevap çiftleri üretmektir.
Çıktıların, hukuki dayanağı olan, anlaşılır ve en önemlisi **OBJEKTİF ve DENGELİ** olmalıdır.

# 1. OBJEKTİFLİK VE İÇERİK KURALLARI (PRIORITY RULES):
1.  **TON:** Cevapları **kesinlikle nötr ve objektif** bir tonda yaz. Tüketicinin her zaman haklı olduğu varsayımından kaçın.
2.  **DENGE:** Q/A çiftlerini üretirken Kanun'un sadece tüketici lehine olan maddelerine değil; **Cayma hakkının istisnaları, satıcının korunması, tüketiciye yüklenen yükümlülükler** gibi dengeleyici hükümlere odaklan.
3.  **KONTEKST ZORUNLULUĞU:** Her cevap, verilen yasal metne (context) doğrudan dayanmalı ve yasal metin ile çelişmemelidir. Context alanındaki bilgi, cevabın temel kaynağıdır.
4.  **DİL:** Soru ve cevaplar, sıradan bir tüketicinin anlayacağı akıcı, günlük Türkçe ile yazılmalıdır (Hukuk dilinden kaçın).
5.  **SORU ÇEŞİTLİLİĞİ:** Gerçek dünya senaryoları, Prosedürel (Nasıl yapılır?), Yansıtıcı (Sonuçları nelerdir?), Kavramsal (Nedir?), Hipotetik ve Sınır Durumları içeren çeşitli soru tipleri kullan.
6.  **FİNANSAL DEĞERLERİN ANONİMİZASYONU VE ARAMASI:**
    İşlenen belgeler miktarı her sene değişen parasal değerler içeriyorsa, bu bilginin cevap kısmında öncelikle bir arama komutu olarak işaretlenmesi ZORUNLUDUR.
    * **Arama Komutu Formatı (Ön İşleme):** Verilen cevapta parasal değerin yerine, modelin arama yapması gerektiğini belirten bir etiket kullanılmalıdır (Örn: **[SEARCH: Tüketici Hakem Heyeti güncel parasal sınırı]**).
    * Context alanı anonimleştirilmeyecek, orijinal metin korunacaktır.
    
# 2. SORU TİPLERİ VE DAĞILIM (QUESTION TYPE DISTRIBUTION):
Ürettiğin sorular şu dağılıma uymalıdır:
- **%30 Prosedürel:** "Nasıl yaparım?", "Ne yapmalıyım?" gibi adım adım rehberlik gerektiren sorular
- **%25 Faktüel:** "Ne zaman?", "Kaç gün?", "Kim?" gibi doğrudan bilgi soruları
- **%20 Hipotetik:** "...olursa ne olur?", "Bu durumda hakkım var mı?" gibi senaryoya dayalı sorular
- **%15 Analitik:** "Neden böyledir?", "Fark nedir?" gibi karşılaştırma ve açıklama gerektiren sorular
- **%10 Yansıtıcı:** "Sonuçları nelerdir?", "Etkileri nedir?" gibi sonuç odaklı sorular

# 3. DENGE İÇİN ÖNCELİK KONULARI (BALANCED CONTENT PRIORITIES):
Soru üretirken şu konulara öncelik ver:
- Cayma hakkı **istisnaları** (özel üretim, bozulabilir ürünler, dijital içerik)
- Tüketicinin **yükümlülükleri** (iade süresi, ürün koşulları, bildirim zorunlulukları)
- Satıcının **hakları** (hatalı iade, zarar tazminatı, sözleşme feshi)
- **Sınır durumları** (ne zaman hak yok, hangi şartlarda istisna)
- **Zaman aşımları** ve prosedürel süreler

# 4. KONTEKST KULLANIM KURALLARI (CONTEXT RULES):
1.  **Standart Kanun Metinlerinde:** Context alanına yalnızca sorunun doğrudan dayandığı **spesifik alt bendi** dahil et (Örn: MADDE 13 - (1) a) bendinin tamamı).
2.  **Makale/Gerekçe Metinlerinde:** Context alanına sorunun dayandığı **ilgili uzun paragrafın tamamını** dahil et.
3.  **Context Doğrulama:** Her cevap, context alanındaki bilgiye dayanmalı. Context'te olmayan bilgi ekleme.

# 5. REFERANS VE ATIF KURALLARI (CITATION RULES):
1.  **CEVAPTA REFERANS:** Cevap metninin içinde, bilginin kaynağını mutlaka belirt.
    * **ÖNEMLİ:** Referans formatı her zaman şu şekilde olmalı: **(Kaynak Adı, m.X/Y)** veya **(Kaynak Adı, m.X/Y-z)**
    * Örnek: (Tüketicinin Korunması Hakkında Kanun, m.15/1-a)
    * Örnek: (Mesafeli Sözleşmeler Yönetmeliği, m.13/2)
    * Makale veya Gerekçe ise sadece kaynak adı: (Akademik Makale) veya (Kanun Gerekçesi)
2.  **KAYNAK TUTARLILIĞı:** 
    * Cevaptaki referans, `source` ve `article` alanlarındaki bilgiyle tam olarak eşleşmeli
    * Her cevap en az bir referans içermeli

# 6. AKICI VE EMPATİK TON ÖRNEKLERİ (TONE EXAMPLES):
Aşağıdaki Soru-Cevap çiftlerini incele ve cevap metninde BİLGİLENDİRİCİ, ANLAYIŞLı ve İNSANCIK TONDAN ayrılma. Cevapları sadece yasal metin değil, bir uzman gibi AKICI ve GÜVEN VEREN bir dille yaz.

## ÖRNEK 1 - Empatik ve Bilgilendirici:
Soru: İnternetten aldığım ürün beklediğim gibi çıkmadı, canım sıkıldı. İade ettiğimde paramı ne zaman alırım?
Cevap: Bu durum can sıkıcı. Endişelenmeyin, cayma hakkınızı kullandığınızda satıcının kanunen 14 takvim günü içinde ödediğiniz tüm parayı (teslimat masrafları dahil) size iade etme zorunluluğu bulunuyor (Mesafeli Sözleşmeler Yönetmeliği, m.15/4). Eğer bu süre içinde ödeme yapmazsa, yasal faiz ödeme yükümlülüğü de doğar.

## ÖRNEK 2 - Dengeli ve Objektif:
Soru: Özel sipariş olarak yaptırdığım ürünü iade edebilir miyim?
Cevap: Maalesef hayır. Tüketicinin özel isteklerine göre hazırlanan veya kişiye özel üretilen mallarda cayma hakkı kullanılamaz (Mesafeli Sözleşmeler Yönetmeliği, m.15/1-b). Bu düzenleme, satıcıyı da korumayı amaçlar çünkü özel üretim ürünler başkasına satılamaz. Ancak üründe ayıp varsa garanti haklarınız devam eder.

## ÖRNEK 3 - Prosedürel ve Rehberlik:
Soru: Satıcı kusurlu hizmet verdi ama oralı olmuyor. Ne yapmalıyım?
Cevap: Öncelikle durumu yazılı olarak (e-posta, ihtarname) belgelemek önemli. Satıcıya 7-10 gün gibi makul bir süre vererek sorunu çözmesini talep edin. Sorun devam ederse, bulunduğunuz yerin Tüketici Hakem Heyeti'ne başvurabilirsiniz (Tüketicinin Korunması Hakkında Kanun, m.68). Başvuru için satış fişi, fatura ve yazışma kayıtlarınızı hazır bulundurun.

## ÖRNEK 4 - Sınır Durumu:
Soru: Aldığım ürünü kullandıktan sonra iade edebilir miyim?
Cevap: Bu durumda dikkat etmeniz gereken önemli bir nokta var: Cayma hakkını kullanırken ürünün ambalajını açıp deneyebilirsiniz, ancak ürünü kullanmış olmamalısınız (Mesafeli Sözleşmeler Yönetmeliği, m.15/3). Eğer ürünü kullandıysanız ve bu kullanım nedeniyle değerinde azalma olduysa, satıcı bu zararı sizden talep edebilir. Sadece deneme amaçlı açmak ile kullanmak arasındaki fark önemlidir.

# 7. SORU BAŞINA CONTEXT MİKTARI:
- Her soru için yalnızca ilgili spesifik madde/bent veya paragraf context olmalı
- Context çok geniş tutulmamalı (max 500 kelime)
- Birden fazla maddeye atıf gerekiyorsa, her biri ayrı Q/A çifti olmalı

# 8. JSON ÇıKTı ŞEMASı (JSON SCHEMA):
Çıktı, aşağıdaki şemaya uygun tek bir JSON Array'i olmalıdır:

[
    {
        "question": "Tüketicinin sorusu (günlük dil, akıcı)",
        "answer": "Uzman cevabı (empatik, dengeli, akıcı, referanslı)",
        "question_type": "Prosedürel / Faktüel / Hipotetik / Analitik / Yansıtıcı",
        "source": "Doküman adı (örn: Tüketicinin Korunması Hakkında Kanun)",
        "context": "İlgili yasal metin veya paragraf (cevabın dayandığı kaynak)",
        "article": "Madde referansı (örn: m.15/1-b) veya null"
    }
]

# 9. KALİTE KONTROL KRİTERLERİ:
Her Q/A çifti şunları sağlamalıdır:
✓ Cevap, context'teki bilgiye doğrudan dayanıyor
✓ Ton empatik ve profesyonel
✓ Hem tüketici hem satıcı perspektifi dengeli
✓ Referans doğru ve açık
✓ Soru gerçekçi bir senaryoyu yansıtıyor
✓ Cevap 100-200 kelime arası (çok kısa veya çok uzun değil)

# 10. YAPILMAMASI GEREKENLER:
❌ Tüketiciyi her zaman haklı gösterme
❌ Hukuki dil kullanma (akıcı, günlük dil kullan)
❌ Context dışı bilgi ekleme
❌ Belirsiz veya muğlak cevaplar
❌ Referanssız iddialar
❌ Aşırı teknik terimler
"""

# ==================== RATE LIMITER ====================
class RateLimiter:
    """API rate limiting yöneticisi"""
    def __init__(self, requests_per_minute=10, initial_delay=6.0):
        self.requests_per_minute = requests_per_minute
        self.min_delay = 60.0 / requests_per_minute  # Minimum istek arası bekleme
        self.current_delay = initial_delay  # Başlangıç gecikmesi
        self.last_request_time = 0
        self.consecutive_errors = 0
        self.max_delay = 60.0  # Maximum bekleme süresi
        
    def wait(self):
        """İstek öncesi bekle"""
        now = time.time()
        time_since_last = now - self.last_request_time
        
        if time_since_last < self.current_delay:
            wait_time = self.current_delay - time_since_last
            print(f"⏳ Rate limiting: {wait_time:.1f}s bekleniyor...")
            time.sleep(wait_time)
        
        self.last_request_time = time.time()
    
    def on_success(self):
        """Başarılı istek sonrası gecikmeyi azalt"""
        self.consecutive_errors = 0
        # Yavaşça normale dön
        self.current_delay = max(self.min_delay, self.current_delay * 0.9)
    
    def on_quota_error(self):
        """Kota hatası sonrası exponential backoff"""
        self.consecutive_errors += 1
        # Exponential backoff: 2^n * base_delay
        backoff_multiplier = min(2 ** self.consecutive_errors, 8)
        self.current_delay = min(self.current_delay * backoff_multiplier, self.max_delay)
        print(f"⚠️ Kota sınırı! Yeni bekleme süresi: {self.current_delay:.1f}s")
        print(f"   Ardışık hata sayısı: {self.consecutive_errors}")
    
    def on_other_error(self):
        """Diğer hatalar için orta seviye artış"""
        self.consecutive_errors += 1
        self.current_delay = min(self.current_delay * 1.5, self.max_delay)

# ==================== PROGRESS TRACKER ====================
class ProgressTracker:
    """İlerleme takip sistemi"""
    def __init__(self, checkpoint_file):
        self.checkpoint_file = checkpoint_file
        self.processed_units = set()
        self.load_checkpoint()
    
    def load_checkpoint(self):
        """Checkpoint dosyasını yükle"""
        if os.path.exists(self.checkpoint_file):
            with open(self.checkpoint_file, 'r') as f:
                data = json.load(f)
                self.processed_units = set(data.get('processed_units', []))
            print(f"✓ Checkpoint yüklendi: {len(self.processed_units)} birim daha önce işlendi")
    
    def save_checkpoint(self):
        """Checkpoint kaydet"""
        with open(self.checkpoint_file, 'w') as f:
            json.dump({
                'processed_units': list(self.processed_units),
                'timestamp': datetime.now().isoformat()
            }, f)
    
    def mark_processed(self, unit_id):
        """Birimi işlenmiş olarak işaretle"""
        self.processed_units.add(unit_id)
        self.save_checkpoint()
    
    def is_processed(self, unit_id):
        """Birim daha önce işlendi mi?"""
        return unit_id in self.processed_units

# ==================== AGENT CLASS (UPDATED) ====================
class Agent:
    def __init__(self, name, role, rate_limiter=None):
        self.name = name
        self.role = role
        self.rate_limiter = rate_limiter or RateLimiter()
        
        # Load API key
        load_dotenv()
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise RuntimeError("Missing GEMINI_API_KEY in environment (.env)")
        
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        
        generation_config = {
            "temperature": 0.7,
            "top_p": 0.95,
            "top_k": 64,
            "max_output_tokens": 16384,
            "response_mime_type": "application/json"
        }
        
        self.model = genai.GenerativeModel(
            "gemini-2.5-flash-lite",
            generation_config=generation_config,
            system_instruction=role
        )
        
        self.total_tokens = 0
        self.input_tokens = 0
        self.output_tokens = 0
        self.input_price = 0.075
        self.output_price = 0.30

    def generate_response(self, prompt):
        """Rate limiting ile API çağrısı"""
        # Rate limiter bekle
        self.rate_limiter.wait()
        
        try:
            response = self.model.generate_content(prompt)
            if not response or not response.text:
                print("generate_response returned an empty or invalid response.")
                self.rate_limiter.on_other_error()
                return None
            
            # Başarılı istek
            self.rate_limiter.on_success()
            
        except Exception as e:
            error_msg = str(e).lower()
            
            # Kota hatalarını yakala
            if any(keyword in error_msg for keyword in ['quota', 'rate limit', 'resource exhausted', '429']):
                print(f"❌ Kota hatası: {e}")
                self.rate_limiter.on_quota_error()
                # Kota hatası sonrası ekstra bekleme
                time.sleep(self.rate_limiter.current_delay)
                return None
            else:
                print(f"❌ API hatası: {e}")
                self.rate_limiter.on_other_error()
                return None

        self.total_tokens += response.usage_metadata.total_token_count
        self.input_tokens += response.usage_metadata.prompt_token_count
        self.output_tokens += response.usage_metadata.candidates_token_count

        return response.text

    def cost(self):
        cost_input = self.input_tokens / 1000000 * self.input_price
        cost_output = self.output_tokens / 1000000 * self.output_price
        cost = cost_input + cost_output

        def format_cost(amount):
            dollars = int(amount)
            cents = (amount - dollars) * 100
            return f"{dollars} dollars {cents:.5f} cents"

        formatted_cost = format_cost(cost)
        formatted_cost_input = format_cost(cost_input)
        formatted_cost_output = format_cost(cost_output)

        print(f"Input tokens: {self.input_tokens} Input Cost: {formatted_cost_input}")
        print(f"Output tokens: {self.output_tokens} Output Cost: {formatted_cost_output}")
        print(f"Total tokens: {self.total_tokens} Total Cost: {formatted_cost}")

    def reset_costs(self):
        self.total_tokens = 0
        self.input_tokens = 0
        self.output_tokens = 0

# ==================== QA AGENT ====================
class QA_Agent(Agent):
    def prepare_QA(self, text, expected_questions=None, source_name=None, article_ref=None):
        """
        expected_questions: Modele kaç soru üretmesi gerektiğini söyler
        source_name: Kaynak adı (örn: "Tüketicinin Korunması Hakkında Kanun")
        article_ref: Madde referansı (örn: "m.15/1-a")
        """
        print("prepare_QA started.")
        
        question_guidance = ""
        if expected_questions:
            question_guidance = f"\n\n**ÖNEMLİ:** Bu metin için tam olarak {expected_questions} adet soru-cevap çifti üret.\n"
        
        reference_guidance = ""
        if source_name and article_ref:
            reference_guidance = f"\n\n**REFERANS FORMATI:** Cevaplarda kaynak gösterirken şu formatı kullan: ({source_name}, {article_ref})\n"
        elif source_name:
            reference_guidance = f"\n\n**REFERANS FORMATI:** Cevaplarda kaynak gösterirken şu formatı kullan: ({source_name})\n"
        
        prompt = f"""
        Sen QA_Agent olarak belirlenen kurallara göre hareket ediyorsun.
        
        Görevin, aşağıdaki yasal metinden yüksek kaliteli soru-cevap çiftleri üretmek.
        
        **Primary role definition'da (QA_role) belirtilen TÜM kurallara SIKI SIKIYA uyman gerekiyor:**
        - Format kuralları (JSON şeması)
        - İçerik kuralları (objektiflik, denge, empatik ton)
        - Referans kuralları (doğru atıf - KAYNAK ADI VE MADDE BİRLİKTE)
        - Kontekst kuralları (context alanı kullanımı)
        - Soru çeşitliliği dağılımı
        {question_guidance}{reference_guidance}
        -------------------------
        
        İŞLENECEK YASAL METİN:
        {text}
        
        -------------------------
        
        Çıktıyı JSON array formatında üret. Her nesne şu alanları içermeli:
        - question
        - answer (MUTLAKA kaynak referansı içermeli)
        - question_type
        - source
        - context
        - article
        """

        print("prepare_QA finished.")
        return self.generate_response(prompt)

# ==================== HELPER FUNCTIONS ====================
def _parse_generated_json(text):
    """JSON parse işlemi - hata toleranslı"""
    if text is None:
        return None
    
    # JSON array'i bul
    start = text.find('[')
    end = text.rfind(']')
    
    if start == -1 or end == -1 or end <= start:
        # Array yoksa, tek obje ara
        start = text.find('{')
        end = text.rfind('}')
        if start == -1 or end == -1 or end <= start:
            raise json.JSONDecodeError("JSON Array/Object not found", text, 0)
        cleaned = "[" + text[start:end+1] + "]"
    else:
        cleaned = text[start:end+1]
    
    return json.loads(cleaned)

def save_csv(filename, qa_list):
    """CSV kaydetme"""
    with open(filename + '.csv', "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["question", "answer", "question_type", "source", "context", "article"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for item in qa_list:
            writer.writerow(item)

# ==================== PDF PROCESSING - ARTICLE BASED ====================
def split_text_into_articles(pages_text):
    """PDF'i madde bazlı parçalara ayırır"""
    full_text = "\n".join(pages_text)
    
    # MADDE pattern'i - Türkçe karakterler dahil
    pattern = re.compile(r"(?m)^MADDE\s+(\d+(?:/[A-ZÇĞİÖŞÜ]+)?)\s*[-–]\s*")
    matches = list(pattern.finditer(full_text))
    
    articles = []
    if not matches:
        return [{"article_no": None, "text": full_text.strip()}]
    
    for idx, match in enumerate(matches):
        start = match.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(full_text)
        article_text = full_text[start:end].strip()
        article_no = match.group(1)
        articles.append({"article_no": article_no, "text": article_text})
    
    return articles

def chunk_long_text(text, max_chars=4000, overlap=200):
    """Uzun metinleri parçalara böler"""
    if len(text) <= max_chars:
        return [text]
    
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

def split_article_into_subclauses(article_text):
    """
    Maddeyi alt bentlere ayırır ve her bentin tam referansını (fıkra + bent) içerir
    
    Türk hukuk metnine uygun format:
    MADDE 3- (1) Metin...
    a) Alt bent...
    b) Alt bent...
    (2) Metin...
    """
    
    # MADDE satırından hemen sonra "- (1)" gibi gelen fıkraları yakala (boşlukları serbest bırak)
    # Örnek: "MADDE 3- (1) Bu Kanunun..."
    article_header_pattern = re.compile(r"MADDE\s+\d+(?:/[A-ZÇĞİÖŞÜ]+)?\s*[-–—]\s*(\(\s*\d+\s*\))", re.UNICODE)
    article_header_match = article_header_pattern.search(article_text)
    
    # Eğer MADDE satırında fıkra varsa, o fıkra numarasını parent olarak kullan
    first_parent = None
    if article_header_match:
        # "( 1 )" gibi ekstra boşlukları da yakalayabilir, burada sadece rakamı al
        first_parent = re.sub(r"\s+", "", article_header_match.group(1)).strip("()")
    
    # Sonraki satırlardaki fıkraları yakala: satır başında veya yeni satırdan sonra (2), (3) gibi
    # Burada da boşlukları ignore et
    subsequent_pattern = re.compile(r"(?<=\n)\s*(\(\s*\d+\s*\))\s+", re.UNICODE)
    subsequent_matches = list(subsequent_pattern.finditer(article_text))
    
    def split_letter_items(text_block, parent_number=None):
        """Harfli bentleri ayırır ve parent fıkra numarasını korur"""
        # a), b), c) gibi bentleri yakala - satır başında veya \n sonrası (boşlukları ignore et)
        letter_pattern = re.compile(r"(?:^|(?<=\n))\s*([a-zçğıöşü])\)\s+", re.MULTILINE | re.UNICODE)
        letter_matches = list(letter_pattern.finditer(text_block))
        
        if not letter_matches:
            # Harfli bent yoksa, sadece fıkra numarasını döndür
            return [{
                "text": text_block.strip(),
                "parent_number": parent_number,
                "letter": None
            }]
        
        items = []
        
        # İlk harfli bentten önce metin varsa, onu da al (fıkra başlangıcı)
        first_letter_start = letter_matches[0].start()
        if first_letter_start > 0:
            intro_text = text_block[:first_letter_start].strip()
            if intro_text:
                items.append({
                    "text": intro_text,
                    "parent_number": parent_number,
                    "letter": None
                })
        
        # Harfli bentleri ayır
        for idx, m in enumerate(letter_matches):
            start = m.start()
            end = letter_matches[idx + 1].start() if idx + 1 < len(letter_matches) else len(text_block)
            letter = m.group(1)
            items.append({
                "text": text_block[start:end].strip(),
                "parent_number": parent_number,
                "letter": letter
            })
        return items
    
    # Eğer hiç numaralı fıkra yoksa
    if not subsequent_matches and not first_parent:
        # Direkt harfli bentleri ara (fıkra numarası yok)
        return split_letter_items(article_text, parent_number=None)
    
    subclauses = []
    
    # İlk fıkrayı işle (MADDE satırındaki)
    if first_parent:
        # MADDE satırından sonraki ilk fıkra bloğunu al
        if subsequent_matches:
            # Bir sonraki fıkraya kadar
            first_block_end = subsequent_matches[0].start()
        else:
            # Metin sonuna kadar
            first_block_end = len(article_text)
        
        # MADDE satırından başlayarak fıkra bloğunu al
        first_block_start = article_header_match.start()
        first_block = article_text[first_block_start:first_block_end].strip()
        
        subclauses.extend(split_letter_items(first_block, parent_number=first_parent))
    
    # Sonraki fıkraları işle
    for idx, m in enumerate(subsequent_matches):
        start = m.start()
        end = subsequent_matches[idx + 1].start() if idx + 1 < len(subsequent_matches) else len(article_text)
        block = article_text[start:end].strip()
        
        # Fıkra numarasını çıkar: "(2)" veya "(  2  )" -> "2"
        group = m.group(1)
        number_str = re.sub(r"\s+", "", group).strip("()")
        
        # Bu fıkra içindeki harfli bentleri ayır
        subclauses.extend(split_letter_items(block, parent_number=number_str))
    
    return subclauses


def extract_full_reference(subclause_data):
    """
    Alt bent verisinden tam referansı oluşturur
    Örnek: {"parent_number": "1", "letter": "a"} -> "1-a"
    Örnek: {"parent_number": "1", "letter": None} -> "1"
    Örnek: {"parent_number": None, "letter": "a"} -> "a"
    """
    parent = subclause_data.get("parent_number")
    letter = subclause_data.get("letter")
    
    if parent and letter:
        return f"{parent}-{letter}"
    elif parent:
        return parent
    elif letter:
        return letter
    else:
        return None


def convert_pdf_to_articles(pdf_name, max_chars=4000, overlap=200, use_subclause_split=True):
    """PDF'i madde/alt bent bazlı parçalara ayırır - GELİŞTİRİLMİŞ REFERANS TAKİBİ"""
    doc = pymupdf.open(pdf_name)
    pages = [page.get_text() for page in doc]
    
    articles = split_text_into_articles(pages)
    units = []
    total_subclauses = 0
    
    for art in articles:
        article_no = art["article_no"]  # "3", "47/A" gibi
        
        if use_subclause_split:
            sub_units = split_article_into_subclauses(art["text"])
        else:
            sub_units = [{
                "text": art["text"],
                "parent_number": None,
                "letter": None
            }]
        
        total_subclauses += len(sub_units)
        
        for sub in sub_units:
            # Alt bent verisi artık dict olarak geliyor
            if isinstance(sub, dict):
                sub_text = sub["text"]
                subclause_ref = extract_full_reference(sub)
            else:
                # Geriye dönük uyumluluk
                sub_text = sub
                subclause_ref = None
            
            # Tam article referansını oluştur
            if article_no and subclause_ref:
                # Örnek: m.3/1-a (fıkra ve bent varsa)
                # Örnek: m.3/1 (sadece fıkra varsa)
                full_article_ref = f"m.{article_no}/{subclause_ref}"
            elif article_no:
                # Sadece madde numarası
                full_article_ref = f"m.{article_no}"
            else:
                full_article_ref = None
            
            # Chunk'lara böl
            pieces = chunk_long_text(sub_text, max_chars=max_chars, overlap=overlap)
            
            # Her parçaya metadata ekle
            for piece in pieces:
                units.append({
                    "text": piece,
                    "article_no": article_no,
                    "subclause_ref": subclause_ref,
                    "full_article_ref": full_article_ref,
                    "parent_number": sub.get("parent_number") if isinstance(sub, dict) else None,
                    "letter": sub.get("letter") if isinstance(sub, dict) else None
                })
    
    print(f"Articles: {len(articles)}, Subclauses: {total_subclauses}, Final units: {len(units)}")
    
    # DEBUG: İlk birkaç birimin referanslarını göster
    print("\n=== İLK 10 BİRİMİN REFERANSLARI ===")
    for i, unit in enumerate(units[:10]):
        ref = unit['full_article_ref']
        preview = unit['text'][:80].replace('\n', ' ')
        print(f"{i+1:2}. {ref:12} | {preview}...")
    print("=" * 80)
    
    return units

# ==================== SOURCE HELPERS ====================
def _tr_lower(text: str) -> str:
    """Türkçe lower case"""
    mapping = str.maketrans({
        "I": "ı", "İ": "i", "Ş": "ş", "Ğ": "ğ",
        "Ü": "ü", "Ö": "ö", "Ç": "ç",
    })
    return text.translate(mapping).lower()

def _tr_upper_first(word: str) -> str:
    """Türkçe title case (ilk harf büyük)"""
    if not word:
        return word
    up_map = {
        "i": "İ", "ı": "I", "ş": "Ş", "ğ": "Ğ",
        "ü": "Ü", "ö": "Ö", "ç": "Ç",
    }
    first = word[0]
    rest = word[1:]
    first_up = up_map.get(first, first.upper())
    return first_up + rest

def turkish_title(text: str) -> str:
    """Türkçe title case"""
    lowered = _tr_lower(text)
    return " ".join(_tr_upper_first(w) for w in lowered.split())

def source_from_filename(pdf_filename: str) -> str:
    """Dosya adından kaynak ismini türet"""
    base = os.path.basename(pdf_filename)
    name_no_ext = os.path.splitext(base)[0]
    
    # Prefix'leri kaldır
    for prefix in ["Regulation_", "Law_", "Guide_", "Paper_"]:
        if name_no_ext.startswith(prefix):
            name_no_ext = name_no_ext[len(prefix):]
            break
    
    spaced = name_no_ext.replace("_", " ")
    return turkish_title(spaced)

# ==================== MAIN PROCESSING FUNCTION ====================
def process_pdfs_in_directory(
    pdf_folder: str,
    source_label: str,
    batch_size: int = 5,
    output_dir: str = None,
    use_subclause_split: bool = True,
    questions_per_unit: int = None
):
    """
    PDF klasöründeki tüm dosyaları işler
    
    Args:
        pdf_folder: PDF klasör yolu
        source_label: Kaynak etiketi (opsiyonel, dosya adından türetilir)
        batch_size: Batch boyutu
        output_dir: Çıktı klasörü
        use_subclause_split: Alt bentlere bölme
        questions_per_unit: Birim başına soru sayısı (None = model karar verir)
    """
    qa_agent = QA_Agent("QA_Agent", QA_role)
    qa_agent.reset_costs()
    
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
    
    pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
    
    for pdf_name in pdf_files:
        qa_list = []
        pdf_path = os.path.join(pdf_folder, pdf_name)
        print(f"\n{'='*80}")
        print(f"Processing PDF: {pdf_name}")
        print(f"{'='*80}")
        
        derived_source = source_from_filename(pdf_name)
        
        # PDF'i birimlere ayır
        units = convert_pdf_to_articles(
            pdf_path,
            max_chars=4000,
            overlap=200,
            use_subclause_split=use_subclause_split
        )
        
        csv_name = pdf_name.split(".")[0] + ".csv"
        
        import concurrent.futures

        def process_unit_with_retries(args):
            qa_agent, unit, unit_idx, i, units, derived_source, questions_per_unit = args

            # Unit artık dict olabilir (metadata ile)
            if isinstance(unit, dict):
                unit_text = unit["text"]
                full_article_ref = unit.get("full_article_ref")
            else:
                unit_text = unit
                full_article_ref = None

            print(f"\nProcessing unit {i + unit_idx + 1}/{len(units)}")
            print(f"Unit length: {len(unit_text)} chars")
            if full_article_ref:
                print(f"Article reference: {full_article_ref}")

            max_retries = 3
            retry_count = 0
            last_error = None

            while retry_count < max_retries:
                try:
                    # Soru üret (source ve article bilgisini ilet)
                    generated_QA = qa_agent.prepare_QA(
                        unit_text,
                        expected_questions=questions_per_unit,
                        source_name=derived_source,
                        article_ref=full_article_ref
                    )

                    if generated_QA is None:
                        print("Generation failed, retrying...")
                        retry_count += 1
                        time.sleep(10)
                        continue

                    # JSON parse
                    generated_QA_JSON = _parse_generated_json(generated_QA)

                    # Context, source ve article alanlarını düzelt
                    for item in generated_QA_JSON:
                        if not item.get("context"):
                            item["context"] = unit_text
                        item["source"] = derived_source

                        # Article referansını full format ile güncelle
                        if full_article_ref:
                            item["article"] = full_article_ref
                        elif not item.get("article"):
                            item["article"] = None

                    print(f"✓ Generated {len(generated_QA_JSON)} questions")
                    return generated_QA_JSON

                except json.JSONDecodeError as e:
                    print(f"JSON decode error: {e}")
                    print(f"Retry {retry_count + 1}/{max_retries}")
                    retry_count += 1
                    last_error = e
                    time.sleep(5)

                except Exception as e:
                    print(f"Unexpected error: {e}")
                    retry_count += 1
                    last_error = e
                    time.sleep(5)

            print(f"⚠ Failed to process unit after {max_retries} retries, skipping...")
            return None

        for i in range(0, len(units), batch_size):
            batch_units = units[i:min(i + batch_size, len(units))]
            batch_num = i // batch_size + 1

            print(f"\n--- Batch {batch_num} started (Units {i+1} to {min(i + batch_size, len(units))}) ---")

            args_list = [
                (qa_agent, unit, unit_idx, i, units, derived_source, questions_per_unit)
                for unit_idx, unit in enumerate(batch_units)
            ]

            batch_results = []
            with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
                futures = [executor.submit(process_unit_with_retries, arg) for arg in args_list]
                for future in concurrent.futures.as_completed(futures):
                    res = future.result()
                    if res:
                        batch_results.extend(res)

            qa_list.extend(batch_results)
            
            # Batch sonuçlarını kaydet
            print(f"\n--- Batch {batch_num} finished ---")
            qa_agent.cost()
            
            out_prefix = f"{csv_name}_batch{batch_num}"
            if output_dir:
                out_prefix = os.path.join(output_dir, out_prefix)
            
            save_csv(out_prefix, qa_list)
            print(f"✓ Saved: {out_prefix}.csv ({len(qa_list)} Q/A pairs)")
            
            qa_list = []
        
        print(f"\n{'='*80}")
        print(f"Finished PDF: {pdf_name}")
        qa_agent.cost()
        print(f"{'='*80}\n")

# ==================== ÖRNEK KULLANIM ====================
if __name__ == "__main__":
    # Çıktı klasörü
    out_root = "/Users/beyzaasan/Projects/HukukPusulasi/outputs"
    
    """# Kanun işleme
    law_dir = "/Users/beyzaasan/Projects/HukukPusulasi/hukukPusulasi-veri/Law"
    law_out = os.path.join(out_root, "Law")
    
    process_pdfs_in_directory(
        pdf_folder=law_dir,
        source_label="TÜKETİCİNİN KORUNMASI HAKKINDA KANUN",
        batch_size=5,
        output_dir=law_out,
        use_subclause_split=True,  # Alt bentlere böl
        questions_per_unit=3  # Birim başına 3 soru (değiştirilebilir)
    )"""
    
    # Diğer kategoriler için benzer şekilde işlem yapılabilir
    # Regulation, Paper, Guide vs.
    # Regulation işleme
    regulation_dir = "/Users/beyzaasan/Projects/HukukPusulasi/hukukPusulasi-veri/Regulation"
    regulation_out = os.path.join(out_root, "Regulation")

    # Klasördeki her PDF dosyası için ayrı ayrı işle, source_label dosya ismi "Regulation_" ile başlıyor ve aradaki alt çizgiler boşluğa çevriliyor
    for pdf_file in os.listdir(regulation_dir):
        if pdf_file.lower().endswith(".pdf"):
            pdf_path = os.path.join(regulation_dir, pdf_file)
            # source_label olarak "Regulation_" sonrası kalan, uzantısız kısmı boşluklu kullan
            base_name = os.path.splitext(pdf_file)[0]
            if base_name.startswith("Regulation_"):
                source_label_part = base_name[len("Regulation_"):]
            else:
                source_label_part = base_name
            source_label = source_label_part.replace("_", " ")
            process_pdfs_in_directory(
                pdf_folder=regulation_dir,
                source_label=source_label,
                batch_size=5,
                output_dir=regulation_out,
                use_subclause_split=True,
                questions_per_unit=3
            )


Processing PDF: Regulation_ABONELİK_SÖZLEŞMELERİ_YÖNETMELİĞİ.pdf
Articles: 28, Subclauses: 99, Final units: 99

=== İLK 10 BİRİMİN REFERANSLARI ===
 1. m.1/1        | MADDE 1 – (1) Bu Yönetmeliğin amacı; abonelik sözleşmelerine ilişkin uygulama us...
 2. m.2/1        | MADDE 2 – (1) Bu Yönetmelik, tüketicinin belirli bir mal veya hizmeti sürekli ve...
 3. m.2/2        | (2) Elektrik, su, doğal gaz ve elektronik haberleşme sektörü dışındaki abonelik ...
 4. m.3/1        | MADDE 3 – (1) Bu Yönetmelik, 7/11/2013 tarihli ve 6502 sayılı Tüketicinin Korunm...
 5. m.4/1        | MADDE 4 – (1) Bu Yönetmeliğin uygulanmasında;...
 6. m.4/1-a      | a) Abonelik sözleşmesi: Tüketicinin belirli bir mal veya hizmeti sürekli veya dü...
 7. m.4/1-b      | b) Bakan: Gümrük ve Ticaret Bakanını,...
 8. m.4/1-c      | c) Elektronik haberleşme: 5/11/2008 tarihli ve 5809 sayılı Elektronik Haberleşme...
 9. m.4/1-ç      | ç) Gecikme zammı oranı: 21/7/1953 tarihli ve 6183 sayılı Amme Alacaklarının Ta

## MAHKEME KARARLARI

In [8]:
import pymupdf
import json
import csv
import os
import time
import re
from dotenv import load_dotenv
import concurrent.futures

# ==================== MAHKEME KARAR ROLE ====================
MahkemeKarar_role = """
# GÖREV TANIMI (PRIMARY ROLE DEFINITION):
Sen, Türk Tüketici Hukuku alanında uzmanlaşmış bir yapay zeka hukuk asistanısın.
Amacın, **hukuki chatbot kullanıcıları** için, mahkeme kararlarından (emsal içtihatlar) Türkçe Soru-Cevap çiftleri üretmektir.
Çıktıların, sıradan vatandaşların anlayacağı, **pratik ve uygulanabilir** olmalıdır.

# 🚨 KRİTİK: CHATBOT İÇİN SORU-CEVAP ÜRETİYORSUN
- Sorular, gerçek insanların chatbot'a soracağı **somut, günlük hayattan** sorular olmalı
- Cevaplar, bir **hukuk danışmanının** vereceği şekilde olmalı
- **ASLA** mahkeme kararını özetleme, onu kullanıcının durumuna **UYGULA**
- **ASLA** "davacı", "davalı", "hüküm", "müvekkil" gibi mahkeme dili kullanma

# 1. SORU OLUŞTURMA KURALLARI (QUESTION RULES):
## ✅ YAPILMASI GEREKENLER:
1. **Somut ve Spesifik:** "Otobüs kazasında yaralandım, masraflarımı kimden talep ederim?"
2. **Gerçek Senaryo:** "İnternetten aldığım telefon bozuk çıktı, ne yapabilirim?"
3. **Net Durum:** "Tatil paketi iptal oldu ama para iadesi alamadım, hakkım var mı?"
4. **Miktar Belirtme:** "... TL değerinde hasar var, sigorta ödemezse ne yaparım?"

## ❌ YAPILMAMASI GEREKENLER:
1. **Muğlak sorular:** "Kasko sigortası ne kadar sorumlu olur?" ❌
2. **Genel kavramlar:** "Mali sorumluluk sigortasının kapsamı nedir?" ❌
3. **Teorik sorular:** "Poliçe limitleri nasıl belirlenir?" ❌
4. **Belirsiz durumlar:** "Tazminat talepleri için ne yapmalıyım?" ❌

# 2. CEVAP YAPISI (4 ADIM - ZORUNLU VE SIKI TAKİP):
## Bu yapıya HARFI HARFINE uymalısın. Her adım ayrı paragraf olmalı.

### ADIM 1: Doğrudan Cevap & Durumu Normalleştirme (2-3 cümle)
- **İLK CÜMLE:** Net cevap ver (Evet/Hayır/Duruma göre değişir)
- **İKİNCİ CÜMLE (İSTEĞE BAĞLI):** Durumun ciddiyetine uygunsa, empati göstererek normalleştir ("Bu tür durumlarla karşılaşmak üzücü/zorlayıcı olabilir.")
- **Örnek 1 (Empati Gerekli):** "Evet, bu durumda masraflarınızı otobüs firmasından talep edebilirsiniz. Kazada yaralanmak ve masraflarla uğraşmak gerçekten zorlayıcı bir süreç."
- **Örnek 2 (Empati Gerekli Değil):** "Evet, bu durumda ödediğiniz ... TL'nin tamamını geri alma hakkınız bulunmaktadır."

### ADIM 2: Emsal Karara Atıf (1 cümle)
- Mahkeme kararını **emsal** olarak göster
- **Örnek:** "Ankara 5. Asliye Ticaret Mahkemesi'nin 2025/287 sayılı kararı benzer bir durumu ele almıştır."
- **ÖNEMLİ:** "Davalı şirket vekili..." gibi ifadeler KULLANMA ❌

### ADIM 3: Hukuki İlkeyi Açıkla (2-3 cümle, günlük dille)
- Mahkemenin kullandığı **PRENSİBİ** açıkla, olayları özetleme
- **Günlük dil** kullan, teknik terimlerden kaçın
- **Örnek:** "Mahkeme kararında; otobüs firmalarının önce zorunlu trafik sigortası ile yolcu zararlarını karşılaması gerektiği belirtilmiştir. Kasko sigortası ancak trafik sigortası yetersiz kalırsa devreye girer."
- **KULLANMA:** "Müvekkil şirket", "Davalı vekili", "Hüküm fıkrası" ❌

### ADIM 4: Kullanıcıya Uygula & Eylem Adımları (2-3 cümle)
- İlkeyi kullanıcının **durumuna bağla**
- **Yapılabilir tavsiye** ver
- **Örnek:** "Sizin durumunuzda da önce otobüs firmasının trafik sigortasına başvurmalısınız. Eğer trafik sigortası tüm masraflarınızı karşılamazsa, o zaman firmanın kasko sigortasından kalan kısmı talep edebilirsiniz. Belge ve fatura gibi masraf kanıtlarınızı saklamanız önemlidir."

# 3. DİL ve TON KURALLARI:
## ✅ KULLANILMALI:
- "Sizin durumunuzda", "Bu durumda", "Benzer bir kararla"
- "Masraflarınızı", "Haklarınızı", "Talebinizi"
- "Yapabilirsiniz", "Talep edebilirsiniz", "Başvurmalısınız"
- "Mahkeme kararında", "Karara göre", "Bu konudaki emsal"

## ❌ KULLANILMAMALI:
- "Davacı", "Davalı", "Müvekkil", "Vekili"
- "Hüküm", "Hüküm fıkrası", "Karar fıkrası"
- "Dilekçe", "Cevap dilekçesi", "Layiha"
- "Teminat limiti tüketildiği hallerde" (çok teknik)
- "Mali sorumluluk sigortası çerçevesinde" (karmaşık)

# 4. SORU TİPLERİ VE DAĞILIM:
- **%70 Practical_Scenario:** Somut, gerçek hayat durumları
  * "Telefon bozuldu, iade edebilir miyim?"
  * "Kazada yaralandım, masrafları kim öder?"
  
- **%15 Legal_Understanding:** Basit kavram açıklamaları (günlük dille)
  * "Cayma hakkı ne demek?"
  * "Zorunlu trafik sigortası ne işe yarar?"
  
- **%15 Negative_Path (ZORUNLU - EN AZ 2 SORU):** * Talebin reddedildiği durumlar
  * "Bilerek kusurlu ürün aldım, iade edebilir miyim?"

# 5. REFERANS VE KAYNAK KURALLARI:
1. **Source Alanı:** Mahkeme adı, karar no, tarih birlikte
   * Format: "ANKARA 5. ASLIYE TICARET MAHKEMESI, 2025/287, 13.03.2025"
   * **ASLA:** "davalarda tüketici mahkemesi" ❌
   
2. **Article Alanı:** Sadece karar numarası
   * Örnek: "2025/287"

# 6. PARAsal DEĞERLERDE ANONİMLEŞTİRME:
- TÜM parasal değerler **... TL** ile değiştirilmeli
- Karar numaraları, tarihler, madde numaraları KORUNMALI

# 7. JSON ÇIKTI ŞEMASı:
[
    {
        "question": "Somut, günlük hayattan soru",
        "answer": "4 adımlı danışman cevabı (SIKI YAPIYA UYGUN)",
        "question_type": "Practical_Scenario / Legal_Understanding / Negative_Path",
        "source": "MAHKEME ADI, KARAR_NO, TARİH",
        "context": "Mahkeme kararının tam metni",
        "article": "KARAR_NO"
    }
]

# 8. KALİTE KONTROL CHECKLİST:
Her Q/A çifti için kontrol et:
✅ Soru somut ve günlük hayattan mı?
✅ Cevap 4 adımlı yapıya uyuyor mu?
✅ "Davacı", "davalı" gibi mahkeme dili YOK mu?
✅ Durumun ciddiyetine uygunsa empati var mı? (1. adım)
✅ Emsal atıf var mı? (2. adım)
✅ İlke açıklaması günlük dille mi? (3. adım)
✅ Kullanıcıya uygulanmış mı? (4. adım)
✅ Source formatı doğru mu? (Mahkeme, Karar No, Tarih)
✅ En az 2 Negative_Path sorusu var mı?
✅ Parasal değerler anonimleştirilmiş mi?

# 9. ÖRNEK Q/A ÇİFTLERİ (DOĞRU FORMAT):

## ✅ MÜKEMMEL ÖRNEK 1 (Practical_Scenario):
{
    "question": "İnternetten aldığım telefon 3 ay içinde 2 kez arızalandı ve her seferinde tamir edilemedi. Ödediğim ... TL'yi geri alabilir miyim?",
    "answer": "Evet, bu durumda ödediğiniz ... TL'nin tamamını geri alma hakkınız bulunmaktadır. Bir ürünün sürekli arızalanması ve tamir edilememesi tüketici için gerçekten can sıkıcı bir durumdur. Yargıtay 13. Hukuk Dairesi'nin 2020/5432 sayılı kararı benzer bir durumu ele almıştır. Mahkeme kararında; ürün aynı arızayı tekrarladığında veya tamir edilemediğinde, TKHK'ya göre artık tüketiciden onarım beklenemeyeceği belirtilmiştir. Sizin durumunuzda da satıcıya yazılı başvuru yaparak paranızın iadesini talep edebilirsiniz. İade talebinizi e-posta veya iadeli taahhütlü mektup ile yapmanız ve arıza raporlarını saklamanız önemlidir.",
    "question_type": "Practical_Scenario",
    "source": "Yargıtay 13. HD, 2020/5432, 15.06.2020",
    "context": "[Kararın tam metni]",
    "article": "2020/5432"
}

## ✅ MÜKEMMEL ÖRNEK 2 (Negative_Path):
{
    "question": "Galeriden 'ağır hasar kayıtlı' olduğunu bilerek ucuza bir araç aldım. 2 ay sonra satmak istediğimde değerinin çok düşük olduğunu fark ettim. Satıcıyı dava edip farkı alabilir miyim?",
    "answer": "Hayır, bu durumda satıcıdan fark talebinde bulunmanız zor görünmektedir. Aracı ucuza almanız ama sonra değer kaybı ile karşılaşmanız üzücü olabilir. Yargıtay 13. Hukuk Dairesi'nin 2019/1234 sayılı kararı tam da bu konuyu ele almıştır. Mahkeme kararında; satın alma sırasında bilinen veya makul olarak bilinmesi gereken kusurlar için satıcının sorumlu tutulamayacağı belirtilmiştir. Sizin durumunuzda, aracın 'ağır hasarlı' olduğunu bilerek ve muhtemelen bu yüzden daha ucuza satın aldığınız için, sonradan bu duruma dayanarak satıcıdan tazminat talep edemezsiniz. Ancak aracın başka gizli kusurları varsa, onlar için hak arayabilirsiniz.",
    "question_type": "Negative_Path",
    "source": "Yargıtay 13. HD, 2019/1234, 10.10.2019",
    "context": "[Kararın tam metni]",
    "article": "2019/1234"
}

## ❌ KÖTÜ ÖRNEK (KULLANMA):
{
    "question": "Kasko sigortasının sorumluluğu ne kadardır?",
    "answer": "Kasko sigortasının sorumluluğu, poliçe limitleri ile sınırlıdır. Davalı şirket vekili, cevap dilekçesinde müvekkili şirketin sorumluluğunun poliçe teminat limitleri dahilinde olacağını belirtmiştir. Mali sorumluluk sigortası teminat limitinin tüketildiği hallerde kasko sorumluluğu başlar...",
    "question_type": "Legal_Understanding",
    "source": "davalarda tüketici mahkemesi, 2024/1578, 31.12.2024",
    "context": "[Metin]",
    "article": "2024/1578"
}
**NEDEN KÖTÜ?**
- Soru muğlak ve genel ❌
- Mahkeme dili kullanılmış (davalı, vekili, müvekkil) ❌
- 4 adımlı yapı yok ❌
- Kullanıcıya uygulanmamış ❌
- Source formatı bozuk ❌

# 10. SON HATIRLATMALAR:
- Her cevap **mutlaka 4 adım** içermeli (4 ayrı paragraf)
- **Asla mahkeme kararını özetleme**, onu kullanıcıya UYGULA
- **Günlük dil** kullan, teknik terimlerden kaçın
- **Somut, spesifik** sorular oluştur
- **Source formatını** doğru yaz (Mahkeme + Karar No + Tarih)
"""

# ==================== RATE LIMITER ====================
class RateLimiter:
    """API rate limiting yöneticisi"""
    def __init__(self, requests_per_minute=10, initial_delay=6.0):
        self.requests_per_minute = requests_per_minute
        self.min_delay = 60.0 / requests_per_minute
        self.current_delay = initial_delay
        self.last_request_time = 0
        self.consecutive_errors = 0
        self.max_delay = 60.0
        
    def wait(self):
        now = time.time()
        time_since_last = now - self.last_request_time
        
        if time_since_last < self.current_delay:
            wait_time = self.current_delay - time_since_last
            print(f"⏳ Rate limiting: {wait_time:.1f}s bekleniyor...")
            time.sleep(wait_time)
        
        self.last_request_time = time.time()
    
    def on_success(self):
        self.consecutive_errors = 0
        self.current_delay = max(self.min_delay, self.current_delay * 0.9)
    
    def on_quota_error(self):
        self.consecutive_errors += 1
        backoff_multiplier = min(2 ** self.consecutive_errors, 8)
        self.current_delay = min(self.current_delay * backoff_multiplier, self.max_delay)
        print(f"⚠️ Kota sınırı! Yeni bekleme süresi: {self.current_delay:.1f}s")
    
    def on_other_error(self):
        self.consecutive_errors += 1
        self.current_delay = min(self.current_delay * 1.5, self.max_delay)

# ==================== AGENT CLASS ====================
class Agent:
    def __init__(self, name, role, rate_limiter=None):
        self.name = name
        self.role = role
        self.rate_limiter = rate_limiter or RateLimiter()
        
        load_dotenv()
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            raise RuntimeError("Missing GEMINI_API_KEY in environment (.env)")
        
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        
        generation_config = {
            "temperature": 0.7,
            "top_p": 0.95,
            "top_k": 64,
            "max_output_tokens": 16384,
            "response_mime_type": "application/json"
        }
        
        self.model = genai.GenerativeModel(
            "gemini-2.5-flash-lite",
            generation_config=generation_config,
            system_instruction=role
        )
        
        self.total_tokens = 0
        self.input_tokens = 0
        self.output_tokens = 0
        self.input_price = 0.075
        self.output_price = 0.30

    def generate_response(self, prompt):
        self.rate_limiter.wait()
        
        try:
            response = self.model.generate_content(prompt)
            if not response or not response.text:
                print("generate_response returned an empty or invalid response.")
                self.rate_limiter.on_other_error()
                return None
            
            self.rate_limiter.on_success()
            
        except Exception as e:
            error_msg = str(e).lower()
            
            if any(keyword in error_msg for keyword in ['quota', 'rate limit', 'resource exhausted', '429']):
                print(f"❌ Kota hatası: {e}")
                self.rate_limiter.on_quota_error()
                time.sleep(self.rate_limiter.current_delay)
                return None
            else:
                print(f"❌ API hatası: {e}")
                self.rate_limiter.on_other_error()
                return None

        self.total_tokens += response.usage_metadata.total_token_count
        self.input_tokens += response.usage_metadata.prompt_token_count
        self.output_tokens += response.usage_metadata.candidates_token_count

        return response.text

    def cost(self):
        cost_input = self.input_tokens / 1000000 * self.input_price
        cost_output = self.output_tokens / 1000000 * self.output_price
        cost = cost_input + cost_output

        def format_cost(amount):
            dollars = int(amount)
            cents = (amount - dollars) * 100
            return f"{dollars} dollars {cents:.5f} cents"

        formatted_cost = format_cost(cost)
        formatted_cost_input = format_cost(cost_input)
        formatted_cost_output = format_cost(cost_output)

        print(f"Input tokens: {self.input_tokens} Input Cost: {formatted_cost_input}")
        print(f"Output tokens: {self.output_tokens} Output Cost: {formatted_cost_output}")
        print(f"Total tokens: {self.total_tokens} Total Cost: {formatted_cost}")

    def reset_costs(self):
        self.total_tokens = 0
        self.input_tokens = 0
        self.output_tokens = 0

# ==================== MAHKEME QA AGENT ====================
class MahkemeQA_Agent(Agent):
    def prepare_QA(self, text, expected_questions=None, source_name=None, karar_no=None):
        """
        expected_questions: Modele kaç soru üretmesi gerektiğini söyler
        source_name: Mahkeme adı (örn: "Yargıtay 13. HD")
        karar_no: Karar numarası (örn: "2023/123")
        """
        print("prepare_QA started.")
        
        question_guidance = ""
        if expected_questions:
            question_guidance = f"\n\n**ÖNEMLİ:** Bu karar için tam olarak {expected_questions} adet soru-cevap çifti üret.\n"
        
        # Sonda referans göstermesine gerek yok, cevapta gerekli yere atıf yapılıyor.
        # O yüzden reference_guidance satırlarını kaldırıyoruz.
        
        prompt = f"""
        Sen MahkemeQA_Agent olarak belirlenen kurallara göre hareket ediyorsun.
        
        Görevin, aşağıdaki mahkeme kararından yüksek kaliteli soru-cevap çiftleri üretmek.
        
        **Primary role definition'da (MahkemeKarar_role) belirtilen TÜM kurallara SIKI SIKIYA uyman gerekiyor:**
        - Format kuralları (JSON şeması)
        - İçerik kuralları (objektiflik, denge, 4 adımlı cevap yapısı)
        - Referans kuralları (doğru atıf - MAHKEME ADI, KARAR NO VE TARİH BİRLİKTE)
        - Kontekst kuralları (context alanı kullanımı)
        - Soru çeşitliliği dağılımı (En az 2 Negative_Path zorunlu)
        - Parasal değerlerin anonimleştirilmesi
        {question_guidance}
        -------------------------
        
        İŞLENECEK MAHKEME KARARI:
        {text}
        
        -------------------------
        
        Çıktıyı JSON array formatında üret. Her nesne şu alanları içermeli:
        - question
        - answer (MUTLAKA kaynak referansı içermeli, 4 adımlı yapı)
        - question_type (en az 2 tanesi Negative_Path olmalı)
        - source (Mahkeme adı ve karar bilgisi)
        - context (İlgili mahkeme kararı metni)
        - article (Karar numarası)
        """

        print("prepare_QA finished.")
        return self.generate_response(prompt)

# ==================== HELPER FUNCTIONS ====================
def _parse_generated_json(text):
    """JSON parse işlemi - hata toleranslı"""
    if text is None:
        return None
    
    start = text.find('[')
    end = text.rfind(']')
    
    if start == -1 or end == -1 or end <= start:
        start = text.find('{')
        end = text.rfind('}')
        if start == -1 or end == -1 or end <= start:
            raise json.JSONDecodeError("JSON Array/Object not found", text, 0)
        cleaned = "[" + text[start:end+1] + "]"
    else:
        cleaned = text[start:end+1]
    
    return json.loads(cleaned)

def save_csv(filename, qa_list):
    """CSV kaydetme - standart format"""
    with open(filename + '.csv', "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["question", "answer", "question_type", "source", "context", "article"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for item in qa_list:
            writer.writerow(item)

# ==================== MAHKEME PDF PROCESSING ====================
def convert_mahkeme_pdf_to_text(pdf_path):
    """
    Mahkeme kararı PDF'ini tek bir metin olarak çıkar
    OCR desteği ile - tam doküman olarak döndürür
    """
    doc = pymupdf.open(pdf_path)
    full_text = ""
    
    for page_num, page in enumerate(doc):
        text = page.get_text()
        
        # Eğer sayfa boşsa, OCR dene
        if len(text.strip()) < 100:
            print(f"Sayfa {page_num + 1} için OCR deneniyor...")
            try:
                pix = page.get_pixmap(dpi=300)
                img_bytes = pix.tobytes("png")
                
                try:
                    from PIL import Image
                    import pytesseract
                    import io
                    
                    img = Image.open(io.BytesIO(img_bytes))
                    text = pytesseract.image_to_string(img, lang='tur')
                    print(f"Sayfa {page_num + 1} OCR ile işlendi: {len(text)} karakter")
                except ImportError:
                    print("OCR için pytesseract ve Pillow kurulu değil.")
            except Exception as e:
                print(f"OCR hatası: {e}")
        
        full_text += text + "\n"
    
    print(f"\nÇıkarılan toplam metin: {len(full_text)} karakter")
    
    if len(full_text.strip()) < 200:
        print("UYARI: PDF'den çok az metin çıkarıldı!")
        return None
    
    return full_text.strip()

def extract_karar_info(text):
    """
    Mahkeme kararından temel bilgileri çıkar
    Returns: (karar_no, mahkeme_adi, tarih)
    """
    karar_no = None
    mahkeme = None
    tarih = None
    
    # Karar numarası pattern'leri (Esas ve Karar numaraları)
    karar_patterns = [
        r'Esas\s*[-–]\s*Karar\s*No\s*[:\.]?\s*(\d{4}/\d+)\s*Esas\s*[-–]\s*(\d{4}/\d+)',  # "2025/287 Esas - 2025/203"
        r'Esas\s*No\s*[:\.]?\s*(\d{4}/\d+)',
        r'Karar\s*No\s*[:\.]?\s*(\d{4}/\d+)',
        r'ESAS\s*NO\s*[:\.]?\s*(\d{4}/\d+)',
        r'KARAR\s*NO\s*[:\.]?\s*(\d{4}/\d+)',
        r'E\.\s*(\d{4}/\d+)',
        r'K\.\s*(\d{4}/\d+)'
    ]
    
    for pattern in karar_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # İlk yakalanan grup karar numarası olsun
            karar_no = match.group(1)
            break
    
    # Mahkeme adı pattern'leri - GENİŞLETİLMİŞ
    mahkeme_patterns = [
        # T.C. ile başlayan mahkemeler
        r'T\.C\.\s+(\w+(?:\s+\w+)*?\s+\d+\.\s+ASLİYE\s+TİCARET\s+MAHKEMESİ)',
        r'T\.C\.\s+(\w+(?:\s+\w+)*?\s+\d+\.\s+ASLİYE\s+HUKUK\s+MAHKEMESİ)',
        r'T\.C\.\s+(\w+(?:\s+\w+)*?\s+TÜKETİCİ\s+MAHKEMESİ)',
        r'T\.C\.\s+(\w+(?:\s+\w+)*?\s+BÖLGE\s+ADLİYE\s+MAHKEMESİ)',
        
        # T.C. olmadan
        r'(\w+\s+\d+\.\s+ASLİYE\s+TİCARET\s+MAHKEMESİ)',
        r'(\w+\s+\d+\.\s+ASLİYE\s+HUKUK\s+MAHKEMESİ)',
        r'(\w+\s+TÜKETİCİ\s+MAHKEMESİ)',
        r'(\w+\s+BÖLGE\s+ADLİYE\s+MAHKEMESİ)',
        
        # Yargıtay
        r'(Yargıtay\s+\d+\.\s*Hukuk\s+Dairesi)',
        r'(YARGITAY\s+\d+\.\s*HUKUK\s+DAİRESİ)',
        
        # Genel pattern - son çare
        r'((?:[A-ZÇĞİÖŞÜ]+\s+)+\d+\.\s+ASLİYE\s+[A-ZÇĞİÖŞÜ]+\s+MAHKEMESİ)',
        r'((?:[A-ZÇĞİÖŞÜ]+\s+)+TÜKETİCİ\s+MAHKEMESİ)'
    ]
    
    for pattern in mahkeme_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            mahkeme = match.group(1).strip()
            # İ harflerini düzelt
            mahkeme = mahkeme.replace('İ', 'İ').replace('I', 'I')
            break
    
    # Tarih pattern'leri - çeşitli formatlar
    tarih_patterns = [
        r'KARAR\s+TARİHİ\s*[:\.]?\s*(\d{2}/\d{2}/\d{4})',
        r'Karar\s+Tarihi\s*[:\.]?\s*(\d{2}\.\d{2}\.\d{4})',
        r'KARAR\s+TARİHİ\s*[:\.]?\s*(\d{2}\.\d{2}\.\d{4})',
        r'(\d{2}\.\d{2}\.\d{4})',  # Genel tarih formatı
        r'(\d{2}/\d{2}/\d{4})'     # Slash ile tarih
    ]
    
    for pattern in tarih_patterns:
        match = re.search(pattern, text)
        if match:
            tarih = match.group(1)
            # Tarihi standart formata çevir (dd.mm.yyyy)
            tarih = tarih.replace('/', '.')
            break
    
    return karar_no, mahkeme, tarih

# ==================== MAIN PROCESSING FUNCTION ====================
def process_mahkeme_pdfs(
    pdf_folder: str,
    batch_size: int = 5,
    output_dir: str = None,
    questions_per_pdf: int = 10
):
    """
    Mahkeme kararlarını toplu olarak işle
    
    Args:
        pdf_folder: PDF klasör yolu
        batch_size: Batch boyutu
        output_dir: Çıktı klasörü
        questions_per_pdf: PDF başına soru sayısı
    """
    qa_agent = MahkemeQA_Agent("MahkemeQA_Agent", MahkemeKarar_role)
    qa_agent.reset_costs()
    
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)
    
    pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]
    
    for pdf_name in pdf_files:
        qa_list = []
        pdf_path = os.path.join(pdf_folder, pdf_name)
        print(f"\n{'='*80}")
        print(f"Mahkeme Kararı İşleniyor: {pdf_name}")
        print(f"{'='*80}")
        
        # PDF'den tam metin çıkar
        full_text = convert_mahkeme_pdf_to_text(pdf_path)
        
        if full_text is None:
            print(f"HATA: {pdf_name} işlenemedi, atlanıyor...")
            continue
        
        # Karar bilgilerini çıkar
        karar_no, mahkeme, tarih = extract_karar_info(full_text)
        
        # Source formatını oluştur
        if mahkeme and karar_no and tarih:
            source_name = f"{mahkeme}, {karar_no}, {tarih}"
        elif mahkeme and karar_no:
            source_name = f"{mahkeme}, {karar_no}"
        elif mahkeme:
            source_name = mahkeme
        else:
            source_name = "Mahkeme Kararı"
        
        print(f"Çıkarılan Bilgiler:")
        print(f"  Mahkeme: {mahkeme}")
        print(f"  Karar No: {karar_no}")
        print(f"  Tarih: {tarih}")
        print(f"  Source: {source_name}")
        
        csv_name = pdf_name.replace(".pdf", ".csv")
        
        print(f"\nTam doküman işleniyor ({len(full_text)} karakter)")
        
        max_retries = 3
        retry_count = 0
        
        while retry_count < max_retries:
            try:
                # Soru üret
                generated_QA = qa_agent.prepare_QA(
                    full_text,
                    expected_questions=questions_per_pdf,
                    source_name=source_name,
                    karar_no=karar_no
                )
                
                if generated_QA is None:
                    print("Kota aşıldı veya üretim başarısız, bekleniyor...")
                    time.sleep(10)
                    retry_count += 1
                    continue
                
                # JSON parse
                generated_QA_JSON = _parse_generated_json(generated_QA)
                
                # Context, source ve article alanlarını düzelt
                for item in generated_QA_JSON:
                    if not item.get("context"):
                        item["context"] = full_text
                    item["source"] = source_name
                    if karar_no and not item.get("article"):
                        item["article"] = karar_no
                
                print(f"✓ {len(generated_QA_JSON)} soru-cevap üretildi")
                qa_list.extend(generated_QA_JSON)
                break
                
            except json.JSONDecodeError as e:
                print(f"JSON decode hatası: {e}")
                retry_count += 1
                if retry_count < max_retries:
                    print(f"Yeniden deneniyor... ({retry_count}/{max_retries})")
                    time.sleep(5)
            
            except Exception as e:
                print(f"Beklenmeyen hata: {e}")
                retry_count += 1
                if retry_count < max_retries:
                    time.sleep(5)
        
        # Sonuçları kaydet
        if qa_list:
            out_prefix = csv_name.replace(".csv", "")
            if output_dir:
                out_prefix = os.path.join(output_dir, out_prefix)
            
            save_csv(out_prefix, qa_list)
            print(f"\n✓ {out_prefix}.csv kaydedildi ({len(qa_list)} soru-cevap)")
        
        print(f"\n{'='*80}")
        print(f"Mahkeme Kararı Tamamlandı: {pdf_name}")
        qa_agent.cost()
        print(f"{'='*80}\n")
    
    print("\n" + "="*80)
    print("TÜM MAHKEME KARARLARI İŞLENDİ")
    print("TOPLAM MALİYET:")
    qa_agent.cost()
    print("="*80)

# ==================== ÖRNEK KULLANIM ====================
if __name__ == "__main__":
    out_root = "/Users/beyzaasan/Projects/HukukPusulasi/outputs"
    mahkeme_dir = "/Users/beyzaasan/Projects/HukukPusulasi/hukukPusulasi-veri/Kararlar"
    mahkeme_out = os.path.join(out_root, "Kararlar")
    
    process_mahkeme_pdfs(
        pdf_folder=mahkeme_dir,
        batch_size=5,
        output_dir=mahkeme_out,
        questions_per_pdf=5  # Her karar için 5 soru
    )


Mahkeme Kararı İşleniyor: 2025_287.pdf
Sayfa 1 için OCR deneniyor...
Sayfa 1 OCR ile işlendi: 1723 karakter
Sayfa 2 için OCR deneniyor...
Sayfa 2 OCR ile işlendi: 2484 karakter
Sayfa 3 için OCR deneniyor...
Sayfa 3 OCR ile işlendi: 818 karakter

Çıkarılan toplam metin: 5028 karakter
Çıkarılan Bilgiler:
  Mahkeme: ANKARA 5. ASLİYE TİCARET MAHKEMESİ
  Karar No: 2025/287
  Tarih: 13.03.2025
  Source: ANKARA 5. ASLİYE TİCARET MAHKEMESİ, 2025/287, 13.03.2025

Tam doküman işleniyor (5026 karakter)
prepare_QA started.
prepare_QA finished.
✓ 7 soru-cevap üretildi

✓ /Users/beyzaasan/Projects/HukukPusulasi/outputs/Kararlar/2025_287.csv kaydedildi (7 soru-cevap)

Mahkeme Kararı Tamamlandı: 2025_287.pdf
Input tokens: 5100 Input Cost: 0 dollars 0.03825 cents
Output tokens: 13890 Output Cost: 0 dollars 0.41670 cents
Total tokens: 18990 Total Cost: 0 dollars 0.45495 cents


Mahkeme Kararı İşleniyor: 2024_1578.pdf
Sayfa 1 için OCR deneniyor...
Sayfa 1 OCR ile işlendi: 569 karakter
Sayfa 2 için OCR d

KeyboardInterrupt: 

## COMBINE CSV BATCH FILES

In [2]:
import glob
import os
import pandas as pd

def combine_csv(folder_path, out_csv_path):
    """
    Belirtilen klasördeki tüm CSV dosyalarını birleştirip, belirtilen yola kaydeder.
    
    Args:
        folder_path (str): CSV dosyalarının bulunduğu klasör.
        out_csv_path (str): Birleştirilen CSV'nin kaydedileceği dosya yolu.
    """
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    df_list = []
    for file in csv_files:
        df = pd.read_csv(file)
        df_list.append(df)
    if df_list:
        # Eğer out_csv_path bir klasörse, dosya adı belirle
        if os.path.isdir(out_csv_path):
            out_csv_file = os.path.join(out_csv_path, "combined.csv")
        else:
            out_csv_file = out_csv_path
        combined_df = pd.concat(df_list, ignore_index=True)
        combined_df.to_csv(out_csv_file, index=False)
        print(f"✓ Combined CSV saved: {out_csv_file} ({len(combined_df)} rows)")
    else:
        print(f"No CSV files found to combine in {folder_path}.")


out_root = "/Users/beyzaasan/Projects/HukukPusulasi/outputs"
# regulations_out = os.path.join(out_root, "Regulation")

# Kullanım örneği
combine_csv(out_root, out_root)  # out_root artık klasör ise çıktıyı combined.csv olarak kaydeder

✓ Combined CSV saved: /Users/beyzaasan/Projects/HukukPusulasi/outputs/combined.csv (12488 rows)
